# Подготовка

## Импорты

In [1]:
from sklearn.preprocessing import LabelEncoder
import numpy as np

from src.dataset import DatasetStore
from src.runner import ExperimentRunner
from src.pipeline import Pipeline
from src.preprocessors import *
from src.models import *
from src.validators import *

In [2]:
import warnings
warnings.filterwarnings('ignore')

## Загрузка данных

In [ ]:
df, schema = DatasetStore.load(
    "data/processed/dataset.csv",
    "data/processed/dataset.schema.json",
    include_missing=False,
)
feature_info = DatasetStore.get_feature_info(
    schema,
    include_missing=False,
)

In [ ]:
columns = ['bled cut', 'albumin']
for col in columns:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
df["diagnosis"] = df["diagnosis"].replace(6, 5)

y = LabelEncoder().fit_transform(df['diagnosis'].values)
X = df.drop(columns=['diagnosis'])

class_labels = [0, 1]

## Инициализация компонентов

In [4]:
onehot_prep = OneHotPreprocessor(
    categorical_features=feature_info['categorical_features'],
    ordinal_features=feature_info['ordinal_features'],
    ordinal_categories=feature_info['ordinal_categories'],
    missing_value=feature_info['missing_value'],
)

In [5]:
onehot_scaled_ord_prep = ScaledOneHotPreprocessor(
    categorical_features=feature_info['categorical_features'],
    ordinal_features=feature_info['ordinal_features'],
    ordinal_categories=feature_info['ordinal_categories'],
    missing_value=feature_info['missing_value'],
    scale_ordinal_features=True,
)

In [6]:
booster_prep = BoosterPreprocessor(missing_value=feature_info['missing_value'])

In [7]:
validator = RepeatedStratifiedKFoldValidator(
    n_splits=3, n_repeats=10, random_state=42,
)

In [8]:
runner = ExperimentRunner(results_dir='_results')

# Обучение

In [ ]:
EXPERIMENTS = [
    {
        'name': 'LogRegGridSearch',
        'model_factory': lambda **p: LogisticRegressionModel(**p),
        'prep': onehot_prep,
        'grid': {
            'C': [0.01, 0.1, 0.5, 1.0, 5.0],
            'solver': ['saga'],
            'class_weight': ['balanced'],
            'max_iter': [200, 300],
        },
    },
    {
        'name': 'SGD_LogLoss',
        'prep': onehot_prep,
        'model_factory': lambda **p: SGDClassifierModel(**p),
        'grid': {
            'loss': ['log_loss', 'modified_huber'],
            'penalty': ['l2', 'l1'],
            'alpha': [0.001, 0.01],
            'class_weight': [None, 'balanced'],
            'learning_rate': ['optimal', 'constant', 'invscaling'],
            'max_iter': [300, 500],
        },
    },
    {
        'name': 'RidgeClassifier',
        'prep': onehot_prep,
        'model_factory': lambda **p: RidgeClassifierModel(**p),
        'grid': {
            'alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
            'class_weight': [None, 'balanced'],
            'fit_intercept': [True],
            'max_iter': [300, 500],
        },
    },
    {
        'name': 'LightGBM',
        'prep': booster_prep,
        'model_factory': lambda **p: LightGBMModel(num_class=num_class, **p),
        'grid': {
            'n_estimators': [100],
            'max_depth': [-1, 3, 5],
            'learning_rate': [0.01, 0.1],
            'num_leaves': [15, 31],
            'subsample': [0.8, 1.0],
            'colsample_bytree': [0.8, 1.0],
            'min_child_samples': [10, 20],
            'early_stopping_rounds': [30],
        },
    },
    {
        'name': 'DecisionTree',
        'prep': onehot_prep,
        'model_factory': lambda **p: DecisionTreeModel(**p),
        'grid': {
            'max_depth': [None, 3, 5, 7, 10],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 5],
            'criterion': ['gini', 'entropy'],
            'class_weight': [None, 'balanced'],
        },
    },
    {
        'name': 'RandomForest',
        'prep': onehot_prep,
        'model_factory': lambda **p: RandomForestModel(**p),
        'grid': {
            'n_estimators': [50, 100],
            'max_depth': [5, 10],
            'min_samples_split': [2, 5],
            'min_samples_leaf': [1, 2, 5],
            'max_features': ['sqrt', 'log2'],
            'criterion': ['gini', 'entropy'],
            'class_weight': [None, 'balanced'],
        },
    },
    {
        'name': 'ComplementNB',
        'prep': onehot_prep,
        'model_factory': lambda **p: ComplementNBModel(**p),
        'grid': {
            'alpha': [0.01, 0.1, 0.5, 1.0, 2.0],
            'norm': [True, False],
            'fit_prior': [True, False],
        },
    },
    {
        'name': 'SVM',
        'prep': onehot_scaled_ord_prep,
        'model_factory': lambda **p: SVMModel(**p),
        'grid': {
            'C': [0.01, 0.1, 1.0, 5.0],
            'kernel': ['rbf', 'linear'],
            'gamma': ['scale', 'auto'],
            'class_weight': ['balanced'],
        },
    },
]


In [ ]:
for exp in EXPERIMENTS:
    print("\n" + "=" * 80)
    print(f"GRIDSEARCH: {exp['name']}")
    print("=" * 80)

    runner.run_gridsearch(
        name=exp['name'],
        preprocessor=exp['prep'],
        model=exp['model_factory'],
        validator=validator,
        param_grid=exp['grid'],
        X=X, y=y,
        class_labels=class_labels,
        scoring_metric='mcc',
    )